# EDA: Survival Patterns by Breast Cancer Subtype

This notebook is my initial look at the METABRIC data — mostly focused on how survival differs across molecular subtypes and what the receptor status breakdown looks like. I also wanted to get a feel for the age distribution and how much data we're actually working with after dropping missing survival info.

The Kaplan-Meier curves were the main thing I wanted to see early on since our first problem statement is specifically about subtype differences. I also added a section on the integrative clusters (IntClust) since those are the main finding from the original Curtis et al. 2012 paper that produced this dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test 
# To install lifelines use ```!pip install lifelines --no-cache-dir```

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

In [ ]:
# load the cleaned METABRIC dataset produced by the preprocessing notebooks
df = pd.read_csv('../data/processed_data/Final_metabric_data.tsv', sep='\t')
print(f'Loaded {df.shape[0]} patients, {df.shape[1]} columns')
df.head(3)

In [ ]:
df.columns

Index(['Patient ID', 'TMB (nonsynonymous)', 'Oncotree Code',
       'Age at Diagnosis', 'Cohort', 'ER Status', 'Tumor Size',
       'Nottingham prognostic index', 'Radio Therapy',
       'Inferred Menopausal State', 'Integrative Cluster', 'Hormone Therapy',
       'HER2 Status', 'HER2 status measured by SNP6',
       'Pam50 + Claudin-low subtype', 'Chemotherapy', 'Type of Breast Surgery',
       'Cellularity', 'Primary Tumor Laterality', 'homdel', 'hetloss', 'gain',
       'amp', 'Relapse Free Status', 'Relapse Free Status (Months)',
       'Overall Survival Status', 'Overall Survival (Months)',
       'Patient's Vital Status'],
      dtype='object')

In [ ]:
# drop rows missing OS data or subtype
df_os = df.dropna(subset=['Overall Survival Status', 'Overall Survival (Months)', 'Pam50 + Claudin-low subtype']).copy()

# encode OS_STATUS as binary (1 = deceased)
df_os['deceased'] = df_os['Overall Survival Status'].str.upper().str.contains('1:DECEASED').astype(int)

print(f'Working with {len(df_os)} patients after dropping missing OS/subtype')
print(f'Deceased: {df_os["deceased"].sum()} ({df_os["deceased"].mean()*100:.1f}%)')

Working with 1980 patients after dropping missing OS/subtype
Deceased: 1143 (57.7%)


## 1. Subtype distribution

CLAUDIN_SUBTYPE gives us the PAM50-like molecular subtypes. Triple-negative corresponds to the Basal-like group here.

In [ ]:
subtype_counts = df_os['Pam50 + Claudin-low subtype'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A. bar chart to show the counts of claudin subtypes in the data. 
subtype_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', len(subtype_counts)), edgecolor='black')
axes[0].set_title('Patient count by molecular subtype', fontsize=14)
axes[0].set_xlabel('Claudin Subtype', fontsize=14)
axes[0].set_ylabel('Number of patients', fontsize=14)
axes[0].tick_params(axis='x', rotation=30, labelsize=14)
axes[0].tick_params(axis='y', labelsize=14)
axes[0].spines['top'].set_visible(True)
axes[0].spines['right'].set_visible(True)
for i, v in enumerate(subtype_counts):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=14)

# B. pie chart
axes[1].pie(subtype_counts.values, labels=subtype_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(subtype_counts)), startangle=45, 
            wedgeprops={'edgecolor': 'white', 'linewidth': 1},
            textprops={'fontsize': 14})

axes[1].set_title('Subtype proportions', fontsize=14)
axes[1].spines['top'].set_visible(True)
axes[1].spines['right'].set_visible(True)

plt.tight_layout()
plt.savefig('../results/claudin_subtype_distribution.png', dpi=300, bbox_inches='tight', 
            pil_kwargs={'compression':'png_lzw'})
plt.show()
print('Luminal A dominates as expected & Basal (TNBC proxy) is ~11%\n')

## 2. Kaplan-Meier survival curves by subtype

The KM curves give us an intuitive view of survival differences before we do any modeling. The log-rank test tells us whether the differences are statistically significant.

In [ ]:
CLAUDIN_SUBTYPE = "Pam50 + Claudin-low subtype"

subtypes = df_os[CLAUDIN_SUBTYPE].unique()
colors = sns.color_palette('Set2', len(subtypes))

fig, ax = plt.subplots(figsize=(11, 7))
kmf = KaplanMeierFitter()

# fit and plot a separate KM survival curve for each molecular subtype
for i, subtype in enumerate(subtypes):
    mask = df_os[CLAUDIN_SUBTYPE] == subtype
    kmf.fit(
        durations=df_os.loc[mask, 'Overall Survival (Months)'],
        event_observed=df_os.loc[mask, 'deceased'],
        label=f'{subtype} (n={mask.sum()})'
    )
    kmf.plot_survival_function(ax=ax, ci_show=False, color=colors[i])

ax.set_title('Overall Survival by Molecular Subtype (METABRIC)', fontsize=14)
ax.set_xlabel('Survival Time (Months)', fontsize=14)
ax.set_ylabel('Survival probability', fontsize=14)
ax.legend(loc='upper right', fontsize=14)
ax.set_ylim(0, 1.05)
ax.tick_params(which='both', labelsize=14)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
plt.tight_layout()
plt.savefig('../results/km_curves_by_subtype.png', dpi=300, bbox_inches='tight', 
            pil_kwargs={'compression':'png_lzw'})
plt.show()

In [ ]:
# log-rank test across all subtypes
results = multivariate_logrank_test(
    df_os['Overall Survival (Months)'],
    df_os[CLAUDIN_SUBTYPE],
    df_os['deceased']
)
print(f'Multivariate log-rank test p-value: {results.p_value:.2e}')
print('Survival differences across subtypes are', 'significant (p < 0.05)' if results.p_value < 0.05 else 'not significant')

Multivariate log-rank test p-value: 2.86e-10
Survival differences across subtypes are significant (p < 0.05)


## 3. Basal (TNBC proxy) vs. everything else

Since problem statement #1 specifically asks about triple-negative patients, I wanted to perform statistical comparison between Basal vs the Rest of the subtypes.

In [ ]:
OS_MONTHS = 'Overall Survival (Months)'
df_os['is_basal'] = (df_os[CLAUDIN_SUBTYPE] == 'Basal').astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
kmf = KaplanMeierFitter()

for group, label, color in [(1, 'Basal / TNBC-like', '#e74c3c'), (0, 'All other subtypes', '#2ecc71')]:
    mask = df_os['is_basal'] == group
    kmf.fit(
        durations=df_os.loc[mask, OS_MONTHS],
        event_observed=df_os.loc[mask, 'deceased'],
        label=f'{label} (n={mask.sum()})'
    )
    kmf.plot_survival_function(ax=ax, ci_show=True, color=color)

# pairwise log-rank
lr = logrank_test(
    df_os.loc[df_os['is_basal']==1, OS_MONTHS],
    df_os.loc[df_os['is_basal']==0, OS_MONTHS],
    event_observed_A=df_os.loc[df_os['is_basal']==1, 'deceased'],
    event_observed_B=df_os.loc[df_os['is_basal']==0, 'deceased']
)

ax.set_title(f'Basal vs. Non-Basal Overall Survival\n(log-rank p = {lr.p_value:.2e})', fontsize=14)
ax.set_xlabel('Survival Time (Months)', fontsize=14)
ax.set_ylabel('Survival probability', fontsize=14)
ax.legend(fontsize=14)
ax.tick_params(which='both', labelsize=14)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
plt.tight_layout()
plt.savefig('../results/km_basal_vs_rest.png', dpi=300, bbox_inches='tight', pil_kwargs={'compression':'png_lzw'})
plt.show()

## 4. Receptor status breakdown

ER/HER2 status is the clinical way subtypes are defined. Thus it is a good idea to explore the counts and how they map onto the molecular subtypes.

In [ ]:
receptor_cols = ['ER Status', 'HER2 Status']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# value counts for each receptor status column, colored by positive/negative/missing
for ax, col in zip(axes, receptor_cols):
    counts = df[col].value_counts(dropna=False)
    colors_bar = ['#e74c3c' if 'Pos' in str(v) else '#95a5a6' if pd.isna(v) else '#3498db' for v in counts.index]
    counts.plot(kind='bar', ax=ax, color=colors_bar, edgecolor='white')
    ax.set_title(col.replace('_', ' '), fontsize=14)
    ax.set_xlabel('')
    ax.tick_params(axis='both', rotation=30, labelsize=14)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    for i, v in enumerate(counts):
        ax.text(i, v + 5, str(v), ha='center', fontsize=14)

plt.suptitle('Receptor Status Distribution (1981 patients)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/receptor_status.png', dpi=300, bbox_inches='tight', pil_kwargs={'compression':'png_lzw'})
plt.show()

In [ ]:
# heatmap: subtype vs. receptor status (% positive)
for col in receptor_cols:
    df_os[col + '_bin'] = (df_os[col].str.upper() == 'POSITIVE').astype(float)

receptor_by_subtype = df_os.groupby(CLAUDIN_SUBTYPE)[[c + '_bin' for c in receptor_cols]].mean() * 100
receptor_by_subtype.columns = ['ER+(%)', 'HER2+(%)']

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(receptor_by_subtype, annot=True, fmt='.0f', cmap='RdYlGn', ax=ax,
            linewidths=0.5, cbar_kws={'label': '% Positive'})
ax.tick_params(axis='both', labelsize=14)
ax.set_title('Receptor Positivity Rate by Molecular Subtype (%)', fontsize=14)
for spine in ax.spines.values():
    spine.set_visible(True)
plt.tight_layout()

plt.savefig('../results/receptor_heatmap.png', dpi=300, bbox_inches='tight', pil_kwargs={'compression':'png_lzw'})
plt.show()
print('Basal subtype shows near-zero ER/HER2 positivity — consistent with TNBC biology')

## 5. Age at diagnosis

Age distribution & its relation to subtypes. Younger age at diagnosis is a known feature of TNBC.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# overall distribution of age
axes[0].hist(df['Age at Diagnosis'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(df['Age at Diagnosis'].median(), color='red', linestyle='--', label=f'Median: {df["Age at Diagnosis"].median():.0f}')
axes[0].tick_params(which='both', labelsize=14)
axes[0].set_title('Age at Diagnosis (all patients)', fontsize=14)
axes[0].set_xlabel('Age', fontsize=14)
axes[0].set_ylabel('Count', fontsize=14)
axes[0].legend(fontsize=14)
axes[0].spines['top'].set_visible(True)
axes[0].spines['right'].set_visible(True)

# boxplot of age by subtype
subtype_order = df_os.groupby(CLAUDIN_SUBTYPE)['Age at Diagnosis'].median().sort_values().index
df_os.boxplot(
    column='Age at Diagnosis', by=CLAUDIN_SUBTYPE, ax=axes[1],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', edgecolor='black'),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black', linewidth=1.5),
    capprops=dict(color='black', linewidth=1.5),
    flierprops=dict(markeredgecolor='black'),
)
axes[1].set_title('Age at Diagnosis by Subtype', fontsize=14)
axes[1].set_xlabel('Subtypes', fontsize=14)
axes[1].set_ylabel('Age', fontsize=14)
axes[1].tick_params(which='both', rotation=45, labelsize=14)
axes[1].spines['top'].set_visible(True)
axes[1].spines['right'].set_visible(True)

plt.tight_layout()
plt.savefig('../results/age_distributions_with_subtypes.png', dpi=300, bbox_inches='tight', pil_kwargs={'compression':'png_lzw'})
plt.show()

## 6. Integrative clusters (IntClust) survival

The original Curtis et al. (2012) *Nature* paper — which defined and published this dataset — identified 10 integrative clusters (IntClust 1–10) using joint copy number + gene expression profiling. These clusters are the paper's central contribution. The INTCLUST column in our data maps directly to them.

It's worth comparing survival across IntClusts directly since the paper shows some stark differences: IntClust 2 (ER+ with 11q13/14 amplification) and IntClust 5 (ERBB2-amplified) have the worst outcomes, while IntClust 3, 4, 7, 8 are relatively favorable.

One interesting note from the paper: Basal-like tumors (concentrated in IntClust 10) have poor *early* survival but show a relative flattening after ~5 years compared to some ER+ groups — worth looking for that crossover in the curves.

In [ ]:
df_intclust = df.dropna(subset=['Overall Survival (Months)', 'Overall Survival Status', 'Integrative Cluster']).copy()
df_intclust['deceased'] = df_intclust['Overall Survival Status'].str.upper().str.contains('DECEASED|DEAD|1').astype(int)
df_intclust['Integrative Cluster'] = df_intclust['Integrative Cluster'].astype(str)

print(f'IntClust analysis: {len(df_intclust)} patients')
print(df_intclust['Integrative Cluster'].value_counts().sort_index())

IntClust analysis: 1980 patients
Integrative Cluster
1       139
10      226
2        72
3       290
4ER+    260
4ER-     83
5       190
6        85
7       190
8       299
9       146
Name: count, dtype: int64


In [ ]:
clusters = sorted(df_intclust['Integrative Cluster'].unique())
colors_ic = sns.color_palette('tab10', len(clusters))

fig, ax = plt.subplots(figsize=(12, 8))
kmf = KaplanMeierFitter()

# fit and plot a KM curve for each integrative cluster
for i, clust in enumerate(clusters):
    mask = df_intclust['Integrative Cluster'] == clust
    kmf.fit(
        durations=df_intclust.loc[mask, 'Overall Survival (Months)'],
        event_observed=df_intclust.loc[mask, 'deceased'],
        label=f'IntClust {clust} (n={mask.sum()})'
    )
    kmf.plot_survival_function(ax=ax, ci_show=True, color=colors_ic[i])

ax.set_title('Overall Survival by Integrative Cluster', fontsize=14)
ax.set_xlabel('Survival Time (months)', fontsize=14)
ax.set_ylabel('Survival probability', fontsize=14)
#ax.set_xlim(0, 180)  # truncate at 15 years like the original paper
ax.legend(loc='upper right', fontsize=14, ncol=2)
ax.set_ylim(0, 1.05)
ax.tick_params(which='both', labelsize=14)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
plt.tight_layout()
plt.savefig('../results/km_by_intclust.png', dpi=300, bbox_inches='tight', pil_kwargs={'compression':'png_lzw'})
plt.show()

## Summary of findings

A few things stand out from this initial look:

- **Luminal A is the largest group** (~38%) and has the best survival, while **Basal (TNBC-like) has the steepest early survival drop** — the KM curves diverge fast in the first ~50 months.
- **Log-rank tests confirm the survival differences are highly significant** across all subtypes and in the Basal vs. non-Basal comparison.
- **Receptor heatmap confirms Basal ≈ TNBC** — near-zero ER/HER2 positivity, consistent with what we'd expect.
- **IntClust curves** — IntClust 2 and 5 are the worst prognosis groups; IntClust 3/4/7/8 are favorable. IntClust 10 (Basal-enriched) shows a noticeable early decline but relatively flat long-term trajectory.

One thing worth flagging for modeling: the paper notes that treatment in METABRIC was **systematic rather than random** — ER+/LN- patients mostly didn't receive chemo, ER-/LN+ patients did, and no HER2+ patient received trastuzumab. So treatment is really a proxy for clinical subtype here, not an independent variable. That complicates any causal claim about treatment effect and we should probably flag it when writing up the adjusted analysis.